In [1]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

#Chemin vers les données sur ma machine
CHEMIN_RAW = '../data/raw'
CHEMIN_INTERIM = '../data/interim'

In [2]:
#---------------------------------------------#
# Paramètres pour lesquels on calcule un rendement
params_rendement = ["DBO5", "DCO", "MES", "NTK", "NH4", "Nglobal", "PT"]


In [3]:
df_Pivot = spark.read.parquet(CHEMIN_INTERIM + '/df_Pivot.parquet')
df_Pivot.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [4]:
# Feature engineering : rendements d'élimination pour 7 polluants principaux. pct d'élimmination
# Formule : (A3 - A4) / A3 × 100 (en %). example: rendement_DBO5 = (DBO5_A3 - DBO5_A4) / DBO5_A3
# rendement = (Entrée − Sortie) / Entrée × 100
# Sécurités : NULL si A3 ou A4 manquant, ou si A3 = 0 (division impossible + valeur non-physique).
# Paramètres exclus : NO2/NO3 (produits intermédiaires du cycle azote), pH/Temp/Volume/Pluie (non-polluants).
# Résultat : 7 nouvelles features métier très informatives pour la classification ET le clustering.

In [5]:
# Construction du dictionnaire de nouvelles colonnes
df_Feat = df_Pivot

for p in params_rendement:
    A3_col = f"A3_{p}"
    A4_col = f"A4_{p}"
    col_rdt = f"Rdt_{p}"
    
    df_Feat = df_Feat.withColumn(
        col_rdt,
        sf.when((
            sf.col(A3_col).isNotNull()) & (sf.col(A3_col) > 0) & (sf.col(A4_col).isNotNull()) # éviter division par 0
            ,(sf.col(A3_col) - sf.col(A4_col)) / sf.col(A3_col) * 100    # en %
               ).otherwise(None)
    )

df_Feat.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [6]:
df_Feat.select(df_Feat.columns[:10]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[10:20]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[20:30]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[30:41]).show(2,truncate = False)

+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
|CdSta       |NmNoSta          |DtPrel    |AnPrel|MoPrel|NoDep|CdTpSta|LbTpSta                  |CapNomSta|A3_Temperature|
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
|030200201000|ACHERY__24920_029|10/01/2020|2020  |01    |02   |29     |STEP collectivité locale |50       |NULL          |
|030200201000|ACHERY__24920_029|12/02/2020|2020  |02    |02   |29     |STEP collectivité locale |50       |NULL          |
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
only showing top 2 rows


+-----+------+-------+------+------+------+------+------+-----+----------+
|A3_pH|A3_MES|A3_DBO5|A3_DCO|A3_NTK|A3_NH4|A3_NO2|A3_NO3|A3_PT|A3_Nglobal|
+-----+------+-------+------+------+------+------+------+-----+----------+
|NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |NULL  |NULL |NULL      |
|NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |NULL  |NULL |NULL      |
+-----+------+-------+------+------+------+------+------+-----+----------+
only showing top 2 rows
+---------+--------+--------------+-----+------+-------+------+------+------+------+
|A3_Volume|A3_Pluie|A4_Temperature|A4_pH|A4_MES|A4_DBO5|A4_DCO|A4_NTK|A4_NH4|A4_NO2|
+---------+--------+--------------+-----+------+-------+------+------+------+------+
|4.35     |4.6     |NULL          |NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |
|2.78     |1.0     |NULL          |NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |
+---------+--------+--------------+-----+------+-------+------+------+------+------+
only showing top

+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+
|A4_NO3|A4_PT|A4_Nglobal|A4_Volume|A4_Pluie|Rdt_DBO5|Rdt_DCO|Rdt_MES|Rdt_NTK|Rdt_NH4|Rdt_Nglobal|
+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+
|NULL  |NULL |NULL      |4.35     |NULL    |NULL    |NULL   |NULL   |NULL   |NULL   |NULL       |
|NULL  |NULL |NULL      |2.78     |NULL    |NULL    |NULL   |NULL   |NULL   |NULL   |NULL       |
+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+
only showing top 2 rows


In [7]:
# description des rendements
df_Feat.select([f"Rdt_{p}" for p in params_rendement]).describe().show(truncate = False)

+-------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|summary|Rdt_DBO5           |Rdt_DCO            |Rdt_MES            |Rdt_NTK            |Rdt_NH4            |Rdt_Nglobal        |Rdt_PT             |
+-------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|count  |91734              |113514             |113507             |75072              |74745              |74130              |75861              |
|mean   |96.74214881626959  |94.40744360957271  |96.05472613790165  |92.18736274444139  |92.59626187750591  |83.64309112611271  |81.83868322034242  |
|stddev |81.63640412301223  |21.496606603799897 |61.50255156087932  |35.483956739660016 |99.72373396113285  |50.51692055952735  |117.54806450550831 |
|min    |-15233.333333333334|-3279.9999999999995|-15354.545454545454|-4654.2359767891685|-18596.7281

In [8]:
# remarques: des anomalies dans les minimums
# compter le nombre de valeurs de rendement négatives

for p in params_rendement:
    colN = f"Rdt_{p}"
    RdtNegCount = df_Feat.select(colN).filter(sf.col(colN)<0).count()
    RdtTotCount = df_Feat.select(f"Rdt_{p}").count()
    print(f"{B}{colN}:{BE}  {RdtNegCount}/{RdtTotCount}")

Rdt_DBO5:  55/2024836


Rdt_DCO:  107/2024836


Rdt_MES:  227/2024836


Rdt_NTK:  225/2024836


Rdt_NH4:  347/2024836


Rdt_Nglobal:  376/2024836


Rdt_PT:  1247/2024836


In [9]:
# Combien de rendements vraiment absurdes (< -100%) ?
for p in params_rendement:
    extreme = df_Feat.filter(sf.col(f"Rdt_{p}") < -100).count()
    total = df_Feat.filter(sf.col(f"Rdt_{p}").isNotNull()).count()
    print(f"Rdt_{p:<8} : {extreme} cas < -100% / {total}")

Rdt_DBO5     : 26 cas < -100% / 91734


Rdt_DCO      : 38 cas < -100% / 113514


Rdt_MES      : 66 cas < -100% / 113507


Rdt_NTK      : 66 cas < -100% / 75072


Rdt_NH4      : 64 cas < -100% / 74745


Rdt_Nglobal  : 67 cas < -100% / 74130


Rdt_PT       : 140 cas < -100% / 75861


In [10]:
# solution pour les rendements absures: clipper à -100, et 100
for p in params_rendement:
    col_rdt = f"Rdt_{p}"
    df_Feat = (
        df_Feat.withColumn(
            col_rdt
            ,sf.when(sf.col(col_rdt) > 100, 100.0)
            .when(sf.col(col_rdt) < -100, -100.0)
            .otherwise(sf.col(col_rdt))
        )
    )
# description des rendements
df_Feat.select([f"Rdt_{p}" for p in params_rendement]).describe().show(truncate = False)

+-------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+
|summary|Rdt_DBO5         |Rdt_DCO          |Rdt_MES          |Rdt_NTK          |Rdt_NH4           |Rdt_Nglobal       |Rdt_PT           |
+-------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+
|count  |91734            |113514           |113507           |75072            |74745             |74130             |75861            |
|mean   |97.40196277353276|94.60443890574865|96.58395438869474|92.76945644794536|93.69828880594474 |83.94748598421486 |82.91333795750477|
|stddev |6.820647232370796|7.695834835089564|9.537480543114622|12.98643547922045|15.231749974505552|17.590349643512276|23.82124895255126|
|min    |-100.0           |-100.0           |-100.0           |-100.0           |-100.0            |-100.0            |-100.0           |
|max    |100.0            |100.0  

In [11]:
# création d'une catégorie pour la capacité nom de la stat: Petite, Moyenne, Grande
### convertir

df_Feat = df_Feat.withColumn('CapNomSta', sf.col('CapNomSta').cast('double'))

df_Feat = (
    df_Feat
    .withColumn(
        'CatSta'
        ,sf.when(sf.col('CapNomSta') < 2000, 'petite')
        .when(sf.col('CapNomSta') < 10000, 'moyenne')
        .otherwise('grande')
    )
)
df_Feat.printSchema()


root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [12]:
df_Feat.select(df_Feat.columns[:10]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[10:20]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[20:30]).show(2,truncate = False)
df_Feat.select(df_Feat.columns[30:42]).show(2,truncate = False)

+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
|CdSta       |NmNoSta          |DtPrel    |AnPrel|MoPrel|NoDep|CdTpSta|LbTpSta                  |CapNomSta|A3_Temperature|
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
|030200201000|ACHERY__24920_029|10/01/2020|2020  |01    |02   |29     |STEP collectivité locale |50.0     |NULL          |
|030200201000|ACHERY__24920_029|12/02/2020|2020  |02    |02   |29     |STEP collectivité locale |50.0     |NULL          |
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+--------------+
only showing top 2 rows


+-----+------+-------+------+------+------+------+------+-----+----------+
|A3_pH|A3_MES|A3_DBO5|A3_DCO|A3_NTK|A3_NH4|A3_NO2|A3_NO3|A3_PT|A3_Nglobal|
+-----+------+-------+------+------+------+------+------+-----+----------+
|NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |NULL  |NULL |NULL      |
|NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |NULL  |NULL |NULL      |
+-----+------+-------+------+------+------+------+------+-----+----------+
only showing top 2 rows


+---------+--------+--------------+-----+------+-------+------+------+------+------+
|A3_Volume|A3_Pluie|A4_Temperature|A4_pH|A4_MES|A4_DBO5|A4_DCO|A4_NTK|A4_NH4|A4_NO2|
+---------+--------+--------------+-----+------+-------+------+------+------+------+
|4.35     |4.6     |NULL          |NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |
|2.78     |1.0     |NULL          |NULL |NULL  |NULL   |NULL  |NULL  |NULL  |NULL  |
+---------+--------+--------------+-----+------+-------+------+------+------+------+
only showing top 2 rows


+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+------+
|A4_NO3|A4_PT|A4_Nglobal|A4_Volume|A4_Pluie|Rdt_DBO5|Rdt_DCO|Rdt_MES|Rdt_NTK|Rdt_NH4|Rdt_Nglobal|Rdt_PT|
+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+------+
|NULL  |NULL |NULL      |4.35     |NULL    |NULL    |NULL   |NULL   |NULL   |NULL   |NULL       |NULL  |
|NULL  |NULL |NULL      |2.78     |NULL    |NULL    |NULL   |NULL   |NULL   |NULL   |NULL       |NULL  |
+------+-----+----------+---------+--------+--------+-------+-------+-------+-------+-----------+------+
only showing top 2 rows


In [13]:
df_Feat.write.mode('overwrite').parquet(CHEMIN_INTERIM + '/df_Feat.parquet')
print("dataframe sauvegardé en Parquet")

dataframe sauvegardé en Parquet


In [14]:
spark.stop()